# 10 — Spring Data

**What you'll learn:**

- Why Spring Data exists — the boilerplate it removes
- **JPA** in one paragraph: the spec, Hibernate as the default implementation
- The `DataSource` and the essential JPA properties
- Mapping classes to tables — `@Entity`, `@Id`, `@GeneratedValue`, `@Column`, `@Table`
- **Relationships** — `@OneToMany`, `@ManyToOne`, `@ManyToMany`, `@OneToOne`
- Cascading, orphan removal, and the owning side
- **Fetch types** — `LAZY` vs `EAGER` and the famous N+1 query problem
- The repository hierarchy — `Repository`, `CrudRepository`, `JpaRepository`
- **Derived query methods** — Spring writes the SQL from the method name
- `@Query` for JPQL and native SQL when derivation isn't enough
- `@Modifying` for write queries
- **Paging and sorting** — `Pageable`, `Page<T>`, `Slice<T>`
- **`@Transactional`** — what it does, propagation, isolation, read-only
- **Specifications** and **Query by Example** as the dynamic-query escape hatches
- **Database migrations** with Flyway (and a Liquibase contrast)

This notebook covers the data layer of every Spring service. The Spring Web layer from notebook 09 is the front door; Spring Data is the inside, where the persistent state lives.

## Why Spring Data

A typical data-access layer, hand-written, repeats the same plumbing in every class: open connection, prepare statement, bind parameters, execute, iterate the result set, map rows to objects, close everything in a `finally`, handle SQL exceptions. Multiply by every table and every query, and you have hundreds of lines of identical, error-prone code.

Spring Data takes that plumbing and reduces it to two things you actually write: an **entity** (what the data looks like in Java) and a **repository interface** (what queries you need). Spring generates the rest at startup using the reflection mechanics from notebook 07.

## JPA in one paragraph

**JPA** — Jakarta Persistence API — is the standard specification for object-relational mapping on the JVM. It defines annotations like `@Entity`, an `EntityManager` interface for CRUD operations, and a query language called **JPQL** (an object-oriented SQL dialect). JPA is just the spec; you need an implementation to use it. **Hibernate** is the de facto implementation that Spring Boot pulls in by default, and Spring Data JPA layers a repository abstraction on top so you rarely touch Hibernate directly.

```text
       Your code
            |
   +------------------+
   |  Spring Data JPA |     repository interfaces, derived queries
   +------------------+
            |
   +------------------+
   |    JPA (spec)    |     @Entity, EntityManager, JPQL
   +------------------+
            |
   +------------------+
   |    Hibernate     |     actual ORM runtime
   +------------------+
            |
   +------------------+
   |     JDBC         |     SQL over the database driver
   +------------------+
            |
       Database
```

## DataSource and the essential properties

Spring Boot autoconfigures a `DataSource` (a connection pool — HikariCP by default) from the standard `spring.datasource.*` properties. JPA itself reads `spring.jpa.*`. A minimal `application.yml`:

In [ ]:
# application.yml
spring:
  datasource:
    url: jdbc:postgresql://localhost:5432/myapp
    username: app
    password: secret
  jpa:
    hibernate:
      ddl-auto: validate        # NEVER use 'update' or 'create' in production
    properties:
      hibernate:
        format_sql: true
    show-sql: false             # set true to see issued SQL in logs

**`ddl-auto: validate`** is the right setting for any non-trivial project. The four options are `none`, `validate` (check schema matches entities), `update` (auto-add missing columns — dangerous; deletes nothing), and `create`/`create-drop` (wipe and rebuild on startup — for tests only). For real schema management, use **Flyway** or **Liquibase** (we'll cover Flyway later in this notebook).

## Entities — mapping a class to a table

An **entity** is a class annotated `@Entity` that JPA tracks as the in-memory representation of a row. Every entity needs an `@Id` field, and JPA also needs a no-arg constructor (which is why we use classes, not records, for entities).

In [ ]:
import jakarta.persistence.*;
import java.time.Instant;

@Entity
@Table(name = "users", indexes = @Index(name = "idx_users_email", columnList = "email"))
public class User {

    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    @Column(name = "email", nullable = false, unique = true, length = 255)
    private String email;

    @Column(nullable = false)
    private String name;

    @Column(name = "created_at", nullable = false, updatable = false)
    private Instant createdAt = Instant.now();

    @Enumerated(EnumType.STRING)
    @Column(nullable = false)
    private Status status = Status.ACTIVE;

    // JPA needs a no-arg constructor
    protected User() {}

    public User(String email, String name) {
        this.email = email;
        this.name = name;
    }

    // getters + (selective) setters
    public Long id() { return id; }
    public String email() { return email; }
    public String name() { return name; }
    public void rename(String newName) { this.name = newName; }
}

A few notes on the mapping:

- **`@GeneratedValue(strategy = IDENTITY)`** — let the database generate the ID (e.g. Postgres `BIGSERIAL`). The other common option is `SEQUENCE` for explicit sequences.
- **`@Column(updatable = false)`** — protect a field from accidental updates (creation timestamps, immutable fields).
- **`@Enumerated(EnumType.STRING)`** — store enums as their *name*, not their ordinal. Ordinal storage breaks the moment you reorder the enum.
- **Records and entities don't mix** — JPA needs to mutate fields after construction (for lazy loading, dirty tracking), which records explicitly disallow. Use a plain class for entities and reserve records for DTOs.

## Relationships

JPA models the four cardinalities of database relationships with four annotations. The crucial concept is the **owning side** — the side that holds the foreign key in the database.

In [ ]:
// Many-to-One — the OWNING side (holds the foreign key)
@Entity
public class Post {
    @Id @GeneratedValue
    private Long id;

    private String title;

    @ManyToOne(fetch = FetchType.LAZY)
    @JoinColumn(name = "author_id", nullable = false)
    private User author;
}

// One-to-Many — the INVERSE side, declared with mappedBy
@Entity
public class User {
    @Id @GeneratedValue
    private Long id;

    @OneToMany(mappedBy = "author", cascade = CascadeType.ALL, orphanRemoval = true)
    private List<Post> posts = new ArrayList<>();
}

// Many-to-Many — needs a join table
@Entity
public class Post {
    @ManyToMany
    @JoinTable(
        name = "post_tags",
        joinColumns = @JoinColumn(name = "post_id"),
        inverseJoinColumns = @JoinColumn(name = "tag_id")
    )
    private Set<Tag> tags = new HashSet<>();
}

// One-to-One — typically with a shared key or unique FK
@Entity
public class User {
    @OneToOne(mappedBy = "user", cascade = CascadeType.ALL)
    private Profile profile;
}

**Cascade** controls what happens to the related entity when you save, update, or delete the owner. `CascadeType.ALL` cascades everything; `CascadeType.PERSIST` and `CascadeType.MERGE` are the most common subset. **`orphanRemoval = true`** deletes child rows when they're removed from the parent's collection.

A repeated piece of advice: **prefer the many-to-one side as the owner** for collections, and keep cascades narrow. Cascade-all-the-things is the source of many "why did deleting a User wipe my entire database" incidents.

## Fetch types and the N+1 problem

Every relationship has a **fetch type** controlling when the related data is loaded.

- **`FetchType.LAZY`** — load on first access. The default for `@OneToMany` and `@ManyToMany`.
- **`FetchType.EAGER`** — load immediately as part of the parent query. The default for `@ManyToOne` and `@OneToOne`.

The default for `@ManyToOne` being EAGER is the source of the most famous JPA performance trap: the **N+1 problem**.

```text
   findAll() runs:                 SELECT * FROM posts        -- 1 query, N rows
                                                            |
   for each post: access post.author triggers:               |
                                   SELECT * FROM users WHERE id = ?   x N
                                                            |
   = 1 + N queries to load N posts and their authors
```

**The fix**: tell Hibernate to fetch the relationship in the original query via a **JOIN FETCH** (in JPQL) or `@EntityGraph`. For everything you can control, **set fetch to LAZY everywhere** and explicitly join when you need the relationship — making the cost visible in the query, not hidden in field access.

In [ ]:
// override the default to be LAZY
@ManyToOne(fetch = FetchType.LAZY)
@JoinColumn(name = "author_id")
private User author;

// in the repository, eagerly fetch the relationship when needed
public interface PostRepository extends JpaRepository<Post, Long> {

    @Query("SELECT p FROM Post p JOIN FETCH p.author WHERE p.publishedAt > :since")
    List<Post> findRecentWithAuthor(@Param("since") Instant since);

    @EntityGraph(attributePaths = {"author", "tags"})
    List<Post> findByPublishedAtAfter(Instant since);
}

## The repository hierarchy

Spring Data exposes a stack of empty marker interfaces. You extend the one that gives you the methods you want — and gain dozens of free methods.

```text
   Repository<T, ID>                <- root marker, no methods
       |
   CrudRepository<T, ID>            <- save, findById, findAll, deleteById, count
       |
   PagingAndSortingRepository<T,ID> <- + findAll(Pageable), findAll(Sort)
       |
   JpaRepository<T, ID>             <- + flush, deleteInBatch, getReferenceById
```

The convention for a Spring Data JPA repository is to extend **`JpaRepository<Entity, IdType>`**. Spring's repository factory generates an implementation at startup — it parses your method names, inspects your `@Query` annotations, and emits the SQL.

In [ ]:
import org.springframework.data.jpa.repository.*;
import org.springframework.data.domain.*;

public interface UserRepository extends JpaRepository<User, Long> {
    // every CRUD method is already here — you write nothing
}

// you get:
// userRepo.save(user);
// userRepo.findById(1L);                  // Optional<User>
// userRepo.findAll();                     // List<User>
// userRepo.findAll(Sort.by("email"));     // sorted
// userRepo.deleteById(1L);
// userRepo.count();

## Derived query methods

The headline feature. Declare a method on the repository interface following a naming convention, and Spring parses the name into a query at startup. No body, no SQL, no annotation.

In [ ]:
public interface UserRepository extends JpaRepository<User, Long> {

    Optional<User> findByEmail(String email);

    List<User> findByStatus(Status status);

    List<User> findByNameContainingIgnoreCase(String fragment);

    List<User> findByCreatedAtAfter(Instant cutoff);

    List<User> findByStatusAndNameStartingWith(Status status, String prefix);

    long countByStatus(Status status);

    boolean existsByEmail(String email);

    // sort and limit are also parts of the name
    List<User> findTop10ByStatusOrderByCreatedAtDesc(Status status);
}

The parser understands a vocabulary of keywords: `findBy`, `countBy`, `existsBy`, `deleteBy`, `Top`/`First`*N*, `OrderBy*Desc/Asc*`, `And`, `Or`, `Between`, `LessThan`, `GreaterThan`, `Like`, `Containing`, `StartingWith`, `EndingWith`, `IgnoreCase`, `In`, `NotIn`, `IsNull`, `IsNotNull`. Once you internalise the vocabulary, you get hundreds of queries for free.

The trade-off: long, multi-clause derived names get hard to read. When a name would have four or more clauses, switch to `@Query` instead.

## `@Query` — when derivation isn't enough

For anything more complex than the derived vocabulary covers, write an explicit query. Spring Data supports JPQL (the object-oriented dialect) and native SQL.

In [ ]:
public interface PostRepository extends JpaRepository<Post, Long> {

    // JPQL — operates on the entity model (Post.author, not posts.author_id)
    @Query("""
        SELECT p FROM Post p
        WHERE p.author.email = :email
          AND p.publishedAt > :since
        ORDER BY p.publishedAt DESC
        """)
    List<Post> recentByAuthor(@Param("email") String email,
                              @Param("since") Instant since);

    // returning a projection — a record interface or class
    @Query("SELECT new com.example.PostSummary(p.id, p.title, p.author.name) FROM Post p")
    List<PostSummary> summaries();

    // native SQL — bypasses JPA's model
    @Query(value = "SELECT * FROM posts WHERE to_tsvector(title) @@ to_tsquery(:q)",
           nativeQuery = true)
    List<Post> fullTextSearch(@Param("q") String query);

    // write queries — need @Modifying, and run inside a transaction
    @Modifying
    @Query("UPDATE Post p SET p.viewCount = p.viewCount + 1 WHERE p.id = :id")
    int incrementViewCount(@Param("id") long id);
}

**JPQL operates on the entity model**, so you traverse fields and relationships (`p.author.email`), not table columns. JPA translates the JPQL into the right SQL for your database dialect. Native SQL is the right escape hatch for database-specific features (full-text search, window functions you can't express in JPQL).

`@Modifying` is required for any query that's not a `SELECT` — it tells JPA to clear the persistence context after the query so subsequent reads see fresh data.

## Paging and sorting

Pass a **`Pageable`** to any query method (derived or `@Query`) and Spring adds `LIMIT`/`OFFSET` + counts the total. The return type **`Page<T>`** includes the page contents plus metadata (total elements, total pages, has-next).

In [ ]:
public interface PostRepository extends JpaRepository<Post, Long> {

    Page<Post> findByAuthorId(Long authorId, Pageable pageable);

    @Query("SELECT p FROM Post p WHERE p.title LIKE %:fragment%")
    Page<Post> searchByTitle(@Param("fragment") String fragment, Pageable pageable);
}

// in a service or controller
Page<Post> page = postRepo.findByAuthorId(
    42L,
    PageRequest.of(0, 20, Sort.by("publishedAt").descending())
);

page.getContent();          // List<Post>
page.getTotalElements();    // long
page.getTotalPages();       // int
page.hasNext();             // boolean

For very large result sets, **`Slice<T>`** skips the expensive `COUNT(*)` query — you lose `getTotalElements`/`getTotalPages` but gain a real performance win. Spring also accepts `Pageable` directly in `@RestController` method signatures: `?page=0&size=20&sort=publishedAt,desc` binds automatically.

## `@Transactional` — what it actually does

A transaction is a unit of database work that either commits as a whole or rolls back as a whole. JPA needs an active transaction for any write (the changes are flushed at commit), and Spring's **`@Transactional`** annotation declares the boundary.

In [ ]:
import org.springframework.transaction.annotation.*;

@Service
public class TransferService {

    private final AccountRepository accountRepo;
    private final LedgerRepository ledgerRepo;

    public TransferService(AccountRepository accountRepo, LedgerRepository ledgerRepo) {
        this.accountRepo = accountRepo;
        this.ledgerRepo = ledgerRepo;
    }

    @Transactional
    public void transfer(long fromId, long toId, long amount) {
        var from = accountRepo.findById(fromId).orElseThrow();
        var to   = accountRepo.findById(toId).orElseThrow();
        from.debit(amount);
        to.credit(amount);
        ledgerRepo.save(new LedgerEntry(fromId, toId, amount));
        // commit on normal return, rollback on RuntimeException
    }

    @Transactional(readOnly = true)
    public List<Account> activeAccounts() {
        return accountRepo.findByStatus(Status.ACTIVE);
    }
}

Three things to know:

- **By default, `@Transactional` rolls back on `RuntimeException` and commits on checked exceptions.** Change with `rollbackFor` / `noRollbackFor`. Modern Java leans unchecked, so the default usually does the right thing.
- **`readOnly = true`** lets the JPA provider skip dirty checking and lets the database use read-only optimisations. Use it on every query method.
- **`@Transactional` works via Spring AOP** (notebook 08). It's a proxy. Self-invocation — one method on a bean calling another `@Transactional` method on the same bean — bypasses the proxy and skips the transaction.

**Propagation** controls what happens when a transactional method is called from inside an existing transaction. Defaults to `REQUIRED` (join the existing one). The other common value is `REQUIRES_NEW` — suspend the outer transaction and start a fresh one — for cases like audit logging that must commit even if the surrounding work rolls back.

## Dynamic queries — Specifications and Query by Example

When the set of filters is decided at runtime (a search UI with optional filters, for example), neither derived methods nor `@Query` fit. Two escape hatches:

- **Specifications** — extend `JpaSpecificationExecutor<T>` on the repository, and compose `Specification<T>` predicates with `and`/`or`. Type-safe but verbose.
- **Query by Example** — build a probe entity with the fields you want to filter on, wrap it in an `Example`, and the framework derives the query.

For most cases you'll outgrow both and reach for **QueryDSL** or **jOOQ**, libraries that give you a fluent, type-safe query builder. Worth knowing they exist; not in scope for this notebook.

## Database migrations with Flyway

Hibernate's `ddl-auto: update` is a footgun in production. The right tool for schema management is a **migration framework** — a tool that runs versioned SQL scripts against your database in order, tracking which ones have been applied. **Flyway** is the dominant choice in the Spring world; **Liquibase** is its main alternative.

The convention for Flyway:

In [ ]:
// src/main/resources/db/migration/V1__create_users.sql
CREATE TABLE users (
    id          BIGSERIAL PRIMARY KEY,
    email       VARCHAR(255) NOT NULL UNIQUE,
    name        VARCHAR(255) NOT NULL,
    created_at  TIMESTAMP NOT NULL DEFAULT now(),
    status      VARCHAR(32) NOT NULL DEFAULT 'ACTIVE'
);

CREATE INDEX idx_users_email ON users(email);

// src/main/resources/db/migration/V2__add_posts.sql
CREATE TABLE posts (
    id            BIGSERIAL PRIMARY KEY,
    author_id     BIGINT NOT NULL REFERENCES users(id),
    title         VARCHAR(255) NOT NULL,
    body          TEXT,
    published_at  TIMESTAMP
);

CREATE INDEX idx_posts_author ON posts(author_id);

Add `flyway-core` to your dependencies, and Spring Boot autoconfigures Flyway. On every startup, Flyway reads the `db/migration/` classpath folder, looks at the `flyway_schema_history` table, and runs any new `V{n}__name.sql` files in order. Migrations are immutable — once applied to any environment, never edit a migration; write a new one.

**The rules of safe schema evolution:**

1. Migrations are append-only — never edit a deployed migration.
2. Make migrations backwards compatible — if a column is going away, first stop writing to it, then deploy, then drop it in a later migration.
3. Keep migrations small and focused — one logical change per file.
4. Test migrations on a copy of production data before deploy.

**Liquibase contrast**: same idea, but migrations are written in XML, YAML, or JSON, with a declarative DSL on top of SQL. Liquibase has stronger rollback support and tracks changes more granularly. Flyway is simpler and more common; Liquibase is preferred in environments that need formal change tracking.

## What's next

You now have the full stack for a production Spring service. Spring Core (notebook 08) wires it together, Spring Web (notebook 09) handles HTTP, and Spring Data (this notebook) persists state. Add Flyway for migrations and `@Transactional` for atomicity and you can write real CRUD services end to end.

Notebook 11 turns to **Spring Boot in Production** — what makes a Spring application *deployable*: auto-configuration, `@ConfigurationProperties` patterns, Actuator endpoints for health and metrics, structured logging, the integrated test slices (`@SpringBootTest`, `@WebMvcTest`, `@DataJpaTest`), Testcontainers for real-DB integration tests, plus messaging via `spring-kafka` and a brief tour of JMS / Spring AMQP. Notebook 12 closes with **Spring Cloud** — distributed-system primitives like config server, service discovery, API gateway, circuit breakers, and Spring Cloud Stream.